# End-to-End Post-Training Quantization Workflow for DeepSeek-V4-Flash Using AMCT

This notebook walks through the AMCT post-training quantization (PTQ) flow for DeepSeek-V4-Flash from baseline evaluation to deployable quantized artifact export.

## Workflow Overview

This notebook covers the following stages:

- Prepare the environment and shared runtime configuration.
- Verify the model path and required checkpoint files.
- Evaluate the baseline BF16 model accuracy.
- Measure the initial loss from direct quantization.
- Extract representative calibration data for each PTQ target.
- Run calibration-based PTQ and save learned quantization parameters.
- Evaluate the calibrated quantized model and export deployable artifacts.

The notebook uses Python cells to build and display the exact CLI commands used by the workflow. When you are ready to execute a step, switch the corresponding `run_cmd(..., dry_run=False)` line back on.


## 1. Environment Setup

This section prepares the Python environment required by the walkthrough. It shows how to install the project dependencies and verify that the key runtime packages can be imported successfully before running any quantization workflow.


In [ ]:
import sys
import subprocess
from pathlib import Path

REPO_ROOT = Path("/data4/l00968832/amct_llm")
REQUIREMENTS_TXT = REPO_ROOT / "requirements.txt"

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Requirements file: {REQUIREMENTS_TXT}")

# Uncomment the next line when the environment still needs dependency installation.
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_TXT)], check=True)

import torch
import transformers

print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print("Environment sanity check passed.")


## 2. Model and Weight Preparation

This section defines the shared runtime paths and experiment settings used throughout the notebook, then checks that the model directory already contains the required checkpoint and tokenizer files before baseline evaluation starts.


In [ ]:
from pathlib import Path
import shlex
import subprocess

REPO_ROOT = Path("/data4/l00968832/amct_llm")
MODEL_PATH = Path("/path/to/deepseek-v4-flash")
CALIB_SOURCE_DIR = Path("/path/to/calibration_or_pileval_source")
OUTPUT_ROOT = REPO_ROOT / "outputs" / "deepseek_v4_flash_walkthrough"
EXPORT_DIR = OUTPUT_ROOT / "export"

MODEL_NAME = "deepseek_v4"
DEVICE = "npu:0"
GRANULARITY = "block"
SEQ_LEN = 4096
NSAMPLES = 128

ALL_QUANT_TARGETS = ["mlp", "moe", "attn-linear", "attn-cache"]
PTQ_TARGETS = ["mlp", "moe", "attn-linear", "attn-cache"]
QUANT_DTYPE = "int"
PTQ_ALGOS = ["lwc", "lac"]

BF16_BIT_CONFIG = REPO_ROOT / "amct_pytorch/configs/bf16.yaml"
W8A8_BIT_CONFIG = REPO_ROOT / "amct_pytorch/configs/w8a8.yaml"

MLP_MOE_PARAM_DIR = OUTPUT_ROOT / "ptq_params" / MODEL_NAME / "mlp_moe"
ATTN_LINEAR_PARAM_DIR = OUTPUT_ROOT / "ptq_params" / MODEL_NAME / "attn_linear"
ATTN_CACHE_PARAM_DIR = OUTPUT_ROOT / "ptq_params" / MODEL_NAME / "attn_cache"

for path in [OUTPUT_ROOT, EXPORT_DIR, MLP_MOE_PARAM_DIR, ATTN_LINEAR_PARAM_DIR, ATTN_CACHE_PARAM_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def pretty_cmd(parts):
    return " \
  ".join(shlex.quote(str(part)) for part in parts)


def show_cmd(parts):
    print(pretty_cmd(parts))


def run_cmd(parts, cwd=REPO_ROOT, dry_run=True):
    print(pretty_cmd(parts))
    if not dry_run:
        subprocess.run([str(part) for part in parts], cwd=str(cwd), check=True)


required_files = [
    MODEL_PATH / "config.json",
    MODEL_PATH / "model.safetensors.index.json",
]

optional_files = [
    MODEL_PATH / "tokenizer.json",
    MODEL_PATH / "tokenizer_config.json",
]

print(f"REPO_ROOT exists: {REPO_ROOT.exists()} -> {REPO_ROOT}")
print(f"MODEL_PATH exists: {MODEL_PATH.exists()} -> {MODEL_PATH}")
print()

for path in required_files:
    print(f"[required] {path}: {path.exists()}")

for path in optional_files:
    print(f"[optional] {path}: {path.exists()}")


## 3. Evaluate Baseline Model Accuracy

This section evaluates the original BF16 model before quantization. The resulting metric is the reference for all later comparisons.


In [ ]:
baseline_dir = OUTPUT_ROOT / "01_baseline_eval"
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_cmd = [
    "python", "amct_pytorch/cli/llm/eval.py",
    "--model", MODEL_PATH,
    "--model_name", MODEL_NAME,
    "--device", DEVICE,
    "--granularity", GRANULARITY,
    "--eval_mode", "bf16",
    "--bit_config", BF16_BIT_CONFIG,
    "--seq_len", str(SEQ_LEN),
    "--output_dir", baseline_dir,
    "--wikitext_final_out", baseline_dir / "wikitext",
]

show_cmd(baseline_cmd)
# run_cmd(baseline_cmd, dry_run=False)


## 4. Apply Direct Quantization and Measure Initial Loss

This section evaluates the model with quant modules enabled but without calibration-based PTQ parameters. It provides the direct quantization reference point before any learned correction is applied.


In [ ]:
direct_quant_dir = OUTPUT_ROOT / "02_direct_quant_eval"
direct_quant_dir.mkdir(parents=True, exist_ok=True)

direct_quant_cmd = [
    "python", "amct_pytorch/cli/llm/eval.py",
    "--model", MODEL_PATH,
    "--model_name", MODEL_NAME,
    "--device", DEVICE,
    "--granularity", GRANULARITY,
    "--eval_mode", "quant",
    "--quant_dtype", QUANT_DTYPE,
    "--bit_config", W8A8_BIT_CONFIG,
    "--seq_len", str(SEQ_LEN),
    "--output_dir", direct_quant_dir,
    "--wikitext_final_out", direct_quant_dir / "wikitext",
    "--quant_target", *ALL_QUANT_TARGETS,
]

show_cmd(direct_quant_cmd)
# run_cmd(direct_quant_cmd, dry_run=False)


## 5. Prepare Representative Calibration Data

This section prepares representative calibration data for the AMCT PTQ workflow.

`extract_ptq_data.py` only supports a single `quant_target` per run, so this notebook builds one command for each target and stores the extracted inputs in separate directories.


In [ ]:
calib_output_root = OUTPUT_ROOT / "03_calibration_data"
calib_output_root.mkdir(parents=True, exist_ok=True)

extract_cmds = {}
for target in PTQ_TARGETS:
    target_output_dir = calib_output_root / target.replace("-", "_")
    target_output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "amct_pytorch/cli/llm/extract_ptq_data.py",
        "--model", MODEL_PATH,
        "--model_name", MODEL_NAME,
        "--device", DEVICE,
        "--granularity", GRANULARITY,
        "--quant_target", target,
        "--data_dir", CALIB_SOURCE_DIR,
        "--output_dir", target_output_dir,
        "--nsamples", str(NSAMPLES),
        "--seq_len", str(SEQ_LEN),
    ]
    extract_cmds[target] = cmd

for target, cmd in extract_cmds.items():
    print(f"\n[{target}]")
    show_cmd(cmd)

# for cmd in extract_cmds.values():
#     run_cmd(cmd, dry_run=False)


## 6. Apply Calibration-Based PTQ with AMCT

This section runs PTQ for each quantization target and saves the learned parameters for later evaluation and export.

`ptq.py` also only supports a single `quant_target` per run. For that reason, the notebook builds one PTQ command per target and writes the resulting parameter files into target-specific directories.


In [ ]:
ptq_output_root = OUTPUT_ROOT / "04_ptq_runs"
ptq_output_root.mkdir(parents=True, exist_ok=True)

param_dir_by_target = {
    "mlp": MLP_MOE_PARAM_DIR,
    "moe": MLP_MOE_PARAM_DIR,
    "attn-linear": ATTN_LINEAR_PARAM_DIR,
    "attn-cache": ATTN_CACHE_PARAM_DIR,
}

ptq_cmds = {}
for target in PTQ_TARGETS:
    target_output_dir = ptq_output_root / target.replace("-", "_")
    target_output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "amct_pytorch/cli/llm/ptq.py",
        "--model", MODEL_PATH,
        "--model_name", MODEL_NAME,
        "--device", DEVICE,
        "--granularity", GRANULARITY,
        "--quant_target", target,
        "--quant_dtype", QUANT_DTYPE,
        "--bit_config", W8A8_BIT_CONFIG,
        "--data_dir", calib_output_root / target.replace("-", "_"),
        "--output_dir", target_output_dir,
        "--seq_len", str(SEQ_LEN),
    ]

    if PTQ_ALGOS:
        cmd.extend(["--algos", *PTQ_ALGOS])

    if target in {"mlp", "moe"}:
        cmd.extend(["--moe_mlp_param_dir", MLP_MOE_PARAM_DIR])
    elif target == "attn-linear":
        cmd.extend(["--attn_linear_param_dir", ATTN_LINEAR_PARAM_DIR])
    elif target == "attn-cache":
        cmd.extend(["--attn_cache_param_dir", ATTN_CACHE_PARAM_DIR])

    ptq_cmds[target] = cmd

for target, cmd in ptq_cmds.items():
    print(f"\n[{target}]")
    show_cmd(cmd)

# for cmd in ptq_cmds.values():
#     run_cmd(cmd, dry_run=False)


## 7. Evaluate Calibrated Quantized Model Accuracy

This section evaluates the quantized model with the learned PTQ parameters loaded from the previous step. The resulting accuracy should be compared against both the BF16 baseline and the direct-quantized reference.


In [ ]:
calibrated_eval_dir = OUTPUT_ROOT / "05_calibrated_quant_eval"
calibrated_eval_dir.mkdir(parents=True, exist_ok=True)

calibrated_eval_cmd = [
    "python", "amct_pytorch/cli/llm/eval.py",
    "--model", MODEL_PATH,
    "--model_name", MODEL_NAME,
    "--device", DEVICE,
    "--granularity", GRANULARITY,
    "--eval_mode", "quant",
    "--quant_dtype", QUANT_DTYPE,
    "--bit_config", W8A8_BIT_CONFIG,
    "--seq_len", str(SEQ_LEN),
    "--output_dir", calibrated_eval_dir,
    "--wikitext_final_out", calibrated_eval_dir / "wikitext",
    "--quant_target", *ALL_QUANT_TARGETS,
    "--moe_mlp_param_dir", MLP_MOE_PARAM_DIR,
    "--attn_linear_param_dir", ATTN_LINEAR_PARAM_DIR,
    "--attn_cache_param_dir", ATTN_CACHE_PARAM_DIR,
]

show_cmd(calibrated_eval_cmd)
# run_cmd(calibrated_eval_cmd, dry_run=False)


## 8. Export Quantized Artifacts for the Inference Repository

This section exports deployable quantized weights, refreshed configuration, and auxiliary files into a standalone output directory that can be consumed by a downstream inference repository.


In [ ]:
export_cmd = [
    "python", "amct_pytorch/cli/llm/deploy.py",
    "--model", MODEL_PATH,
    "--model_name", MODEL_NAME,
    "--device", DEVICE,
    "--granularity", GRANULARITY,
    "--quant_target", *ALL_QUANT_TARGETS,
    "--quant_dtype", QUANT_DTYPE,
    "--bit_config", W8A8_BIT_CONFIG,
    "--output_dir", EXPORT_DIR,
    "--moe_mlp_param_dir", MLP_MOE_PARAM_DIR,
    "--attn_linear_param_dir", ATTN_LINEAR_PARAM_DIR,
    "--attn_cache_param_dir", ATTN_CACHE_PARAM_DIR,
]

show_cmd(export_cmd)
# run_cmd(export_cmd, dry_run=False)
